# FX Graph Mode Post-Training Quantization 

In [ ]:
import os
import torch

from model_compression.src.utils import load_data
from model_compression.src.Quantization.utils.post_training_quantization import quantize_model
from model_compression.src.Quantization.utils.preprocessing import load_kd_model
from model_compression.src.utils import calculate_metrics
from model_compression.src.utils import test_inference
from model_compression.src.utils import model_size, measure_inference_performance

In [ ]:
# Specify random seed for repeatable results
_ = torch.manual_seed(191009)

### Helper Functions

In [ ]:
def load_quantized_model(model_path: str, device: torch.device = torch.device("cpu")) -> torch.jit.ScriptModule:
    """
    Loads a quantized TorchScript model from the specified file.

    Parameters:
        model_path (str): Path to the saved TorchScript model (.pt file).
        device (torch.device): The device on which to load the model. Defaults to CPU.
    
    Returns:
        torch.jit.ScriptModule: The loaded quantized model.
    """
    # Load the TorchScript model from disk, mapping it to the specified device.
    model = torch.jit.load(model_path, map_location=device)
    # Set the model to evaluation mode (important for inference)
    model.eval()
    return model


### Main

In [ ]:
# Paths for the dataset and model weights
dataset = "SkinCancer"  # Update with your dataset path
batch_size = 32

dataloaders = load_data(dataset=dataset, batch_size=batch_size)
num_classes = len(dataloaders["train"].dataset.classes)

model_weights_path = 'models/SkinCancer/mobilenet_v2_best_model.pth'  # Update with your saved model weights

# Device configuration - quantization is often done on CPU.
device = torch.device("cpu")

# Load the teacher model (with fine-tuned weights).
original_model = load_kd_model("mobilenet_v2", model_weights_path, num_classes)

# Choose an example input from the training data (we only need the image, not the label)
example_input = next(iter(dataloaders["train"]))[0].to(device)

# Perform quantization
quantized_model = quantize_model(original_model, example_input, dataloaders["test"])

# Optionally, save the quantized model using TorchScript for later deployment
quantized_model_scripted = torch.jit.script(quantized_model)
save_path = os.path.join(os.path.dirname(model_weights_path), "mobilenetv2_quantized.pt")
torch.jit.save(quantized_model_scripted, save_path)
print(f"Quantized model saved to: {save_path}")

### Benchmarks

In [ ]:
# Print the size of the float model
print("Float model size:")
model_size(original_model)

# Print the size of the quantized model
print("Quantized model size:")
model_size(quantized_model)